# Player Lookalikes - incremental update

Re-uses the work already done in `player_lookalikes.ipynb` (which is left untouched):

- `../data/player_price_matches.csv` - the name matches you confirmed by hand (FPL name vs. match-data name)
- `../data/player_lookalike.csv` - the lookalikes already chosen

Only (player, season) pairs that are **not** in those files are processed (i.e. the new 25-26 players).
Outputs go to `*_v2.csv` files so the originals are never overwritten. Once happy, swap them in.

In [ ]:
import pandas as pd
import random as rand
from difflib import SequenceMatcher

# Existing work (read only)
OLD_MATCHES = '../data/player_price_matches.csv'
OLD_LOOKALIKES = '../data/player_lookalike.csv'

# New outputs (old + new combined)
NEW_MATCHES = '../data/player_price_matches_v2.csv'
NEW_LOOKALIKES = '../data/player_lookalike_v2.csv'

## 1. Build player_data (same as before, but via a season lookup)

In [ ]:
fantasy = pd.read_csv('../data/master/master_fantasy.csv')

# (first match_id, last match_id, season)
season_ranges = [
    (918893,  919272,  '17-18'),
    (987592,  987971,  '18-19'),
    (1059702, 1060081, '19-20'),
    (2128286, 2128665, '20-21'),
    (2210271, 2210650, '21-22'),
    (2292810, 2293189, '22-23'),
    (2367538, 2367917, '23-24'),
    (2444470, 2444849, '24-25'),
    (2561895, 2562274, '25-26'),
    # add 26-27 here once it exists in master_fantasy
]

player_data = fantasy[['match_id', 'nick_name', 'full_name', 'pos', 'team', 'cost']].copy()
player_data['season'] = 'none'
for lo, hi, season in season_ranges:
    player_data.loc[player_data.match_id.between(lo, hi), 'season'] = season

assert (player_data.season == 'none').sum() == 0, 'some match_ids are outside the season ranges'
player_data.season.value_counts().sort_index()

## 2. Initial prices from vaastav

In [ ]:
vaastav_seasons = ['17-18', '18-19', '19-20', '20-21', '21-22', '22-23', '23-24', '24-25', '25-26', '26-27']

initial_costs = pd.DataFrame()
for season in vaastav_seasons:
    table = pd.read_csv(f'../data/Vastaav/players_raw_{season.replace("-", "_")}.csv')
    temp = table[['first_name', 'second_name', 'cost_change_start', 'now_cost']].copy()
    temp['full_name'] = temp['first_name'] + ' ' + temp['second_name']
    temp['initial_cost'] = (temp['now_cost'] - temp['cost_change_start']) / 10
    temp['season'] = season
    initial_costs = pd.concat([initial_costs, temp[['full_name', 'initial_cost', 'season']]], ignore_index=True)

initial_costs = initial_costs[initial_costs.initial_cost >= 4.0]
initial_costs.season.value_counts().sort_index()

## 3. Name matching - only the NEW unmatched names

Load the matches you already confirmed, then find (name, season) pairs that still don't match and aren't in that file.

In [ ]:
old_matches = pd.read_csv(OLD_MATCHES)
print(f'{len(old_matches)} existing matches loaded, seasons: {sorted(old_matches.season.unique())}')

merged = pd.merge(player_data, initial_costs, on=['full_name', 'season'], how='outer', indicator=True)

# only consider seasons we actually have match data for
merged = merged[merged.season.isin(player_data.season.unique())]

left = (merged[merged._merge == 'left_only']
        .drop_duplicates(subset=['full_name', 'season'])
        .sort_values(by='match_id')[['full_name', 'season']])
right = (merged[merged._merge == 'right_only']
         .drop_duplicates(subset=['full_name', 'season'])
         .sort_values(by=['full_name', 'season'])[['full_name', 'season']])

# drop anything already handled in the old notebook
done = set(zip(old_matches.name1, old_matches.season))
left_new = left[[(n, s) not in done for n, s in zip(left.full_name, left.season)]].reset_index(drop=True)

print(f'{len(left)} unmatched total, {len(left_new)} are new and need a decision')
left_new.season.value_counts()

In [ ]:

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio() * 100

def top_matches(name, season, right, top_n=5):
    cands = right[right.season == season].copy()
    cands['score'] = cands.full_name.map(lambda n: similarity(name, n))
    return cands.sort_values('score', ascending=False).head(top_n).reset_index(drop=True)

# kept between runs: if the cell stops, just re-run it and it carries on where it left off
if 'new_matches' not in globals():
    new_matches = []

decided = {(m['name1'], m['season']) for m in new_matches}

for i, (name1, season1) in enumerate(left_new[['full_name', 'season']].itertuples(index=False), 1):
    if (name1, season1) in decided:
        continue
    cands = top_matches(name1, season1, right)
    options = '  |  '.join(f"{j + 1}) {c.full_name} ({c.score:.0f})" for j, c in cands.iterrows())
    print(f"[{i}/{len(left_new)}] {name1} ({season1}): {options}", flush=True)
    # VS Code's input box only reliably shows a one-line prompt, so the options go in it too
    response = input(f"[{i}/{len(left_new)}] {name1} ({season1}) ->  {options}  |  Enter = none, q = stop: ").strip().lower()
    if response == 'q':
        break
    if response.isdigit() and 1 <= int(response) <= len(cands):
        c = cands.iloc[int(response) - 1]
        new_matches.append({'name1': name1, 'name2': c.full_name, 'season': season1,
                            'score': c.score, 'match_indicator': 1})
    else:
        # record as "no match" so we never ask again
        new_matches.append({'name1': name1, 'name2': None, 'season': season1,
                            'score': 0, 'match_indicator': 0})

new_matches_df = pd.DataFrame(new_matches)
print(f"\n{len(new_matches_df)} decided, {len(left_new) - len(new_matches_df)} still to do")


In [ ]:
matches_df = pd.concat([old_matches, new_matches_df], ignore_index=True)
matches_df.to_csv(NEW_MATCHES, index=False)
print(f'saved {len(matches_df)} matches ({len(new_matches_df)} new) -> {NEW_MATCHES}')

## 4. Apply initial costs to player_data

In [ ]:
player_data = player_data.merge(initial_costs, on=['full_name', 'season'], how='left')

# fill the name-mismatched players from the confirmed matches (indicator == 1 only)
good = matches_df[matches_df.match_indicator == 1].merge(
    initial_costs.rename(columns={'full_name': 'name2', 'initial_cost': 'matched_cost'}),
    on=['name2', 'season'], how='left'
).drop_duplicates(subset=['name1', 'season'])

player_data = player_data.merge(
    good[['name1', 'season', 'matched_cost']].rename(columns={'name1': 'full_name'}),
    on=['full_name', 'season'], how='left'
)
player_data['initial_cost'] = player_data.initial_cost.fillna(player_data.matched_cost)
player_data = player_data.drop(columns='matched_cost')

still_missing = player_data[player_data.initial_cost.isna()].drop_duplicates(subset=['full_name', 'season'])
print(f'{len(still_missing)} (player, season) pairs still have no initial cost')
still_missing.season.value_counts().sort_index()

## 5. Dates and appearance counts

In [ ]:
team_stats_df = pd.read_csv('../data/master/master_team_stats.csv')

def date_transform(date):
    for i, m in enumerate(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], 1):
        date = date.replace(m, str(i))
    for suffix in ['th', 'st', 'nd', 'rd']:
        date = date.replace(suffix, '')
    return date

team_stats_df['date'] = pd.to_datetime(team_stats_df['date'].apply(date_transform), format='%d %m %Y')
match_id_date_map = team_stats_df.drop_duplicates('match_id').set_index('match_id')['date']

player_data['date'] = player_data.match_id.map(match_id_date_map)
player_data = player_data.drop_duplicates(subset=['full_name', 'season', 'match_id'], keep='first')
player_data = player_data.sort_values(by=['date']).reset_index(drop=True)
player_data = player_data.drop(columns=['match_id', 'cost'])

# appearance counting (vectorised version of the old iterrows loop)
n = player_data.groupby('full_name').cumcount() + 1
player_data['first_appearance'] = player_data.groupby('full_name').date.transform('min')
nineteenth_date = player_data[n == 19].drop_duplicates('full_name').set_index('full_name')['date']
player_data['nineteenth_appearance'] = player_data.full_name.map(nineteenth_date).where(n >= 19, pd.NaT)

player_data.tail()

## 6. Which players need a lookalike - and which already have one

In [ ]:
player_matches_needed = (player_data[(player_data.date >= '2018-08-01') & (player_data.nineteenth_appearance.isna())]
                         .copy().drop_duplicates(subset=['full_name']))

old_lookalikes = pd.read_csv(OLD_LOOKALIKES)
have = set(zip(old_lookalikes.full_name, old_lookalikes.season))

needed_new = player_matches_needed[[(n, s) not in have for n, s in zip(player_matches_needed.full_name, player_matches_needed.season)]].copy()

print(f'{len(player_matches_needed)} players need lookalikes in total')
print(f'{len(old_lookalikes)} already done, {len(needed_new)} new')
needed_new.season.value_counts()

## 7. Lookalike function (logic unchanged)

Note that it picks randomly between equally-priced candidates, which is exactly why the old picks are loaded from the CSV rather than regenerated.

In [ ]:
season_changes = {
    "18-19": {"promoted": ["WOL", "CAR", "FUL"], "relegated": ["SWA", "STK", "WBA"]},
    "19-20": {"promoted": ["NOR", "SHU", "AVL"], "relegated": ["CAR", "FUL", "HUD"]},
    "20-21": {"promoted": ["LEE", "WBA", "FUL"], "relegated": ["BOU", "WAT", "NOR"]},
    "21-22": {"promoted": ["NOR", "WAT", "BRE"], "relegated": ["FUL", "WBA", "SHU"]},
    "22-23": {"promoted": ["FUL", "BOU", "NFO"], "relegated": ["BUR", "WAT", "NOR"]},
    "23-24": {"promoted": ["BUR", "SHU", "LUT"], "relegated": ["LEI", "LEE", "SOU"]},
    "24-25": {"promoted": ["LEI", "IPS", "SOU"], "relegated": ["LUT", "BUR", "SHU"]},
    "25-26": {"promoted": ["LEE", "BUR", "SUN"], "relegated": ["LEI", "IPS", "SOU"]},
    # "26-27": {"promoted": [...], "relegated": [...]},   # fill in when needed
}

def find_lookalike(player_name, team, position, season, cost, date):
    if pd.isna(cost):
        print(f"No initial cost for {player_name} ({season}) - finish the name matching first")
        return None

    # newly promoted teams borrow from the team that went down
    if team in season_changes[season]["promoted"]:
        team = season_changes[season]["relegated"][season_changes[season]["promoted"].index(team)]

    prev_season = f"{int(season[:2])-1}-{int(season[3:5])-1:02d}"

    similar_players = player_data[
        (player_data['team'] == team) &
        (player_data['pos'] == position) &
        ((player_data['season'] == season) | (player_data['season'] == prev_season)) &
        (player_data['full_name'] != player_name) &
        (player_data['nineteenth_appearance'] < pd.to_datetime(date))
    ].copy()

    # current price if possible, otherwise last season's
    similar_players = similar_players.sort_values(by=['season'], ascending=False).drop_duplicates(subset=['full_name'], keep='first')

    if similar_players.empty:
        print(f"No valid lookalikes found for {player_name} ({season})")
        return None

    similar_players['cost_diff'] = similar_players['initial_cost'] - cost
    similar_players['cost_diff_abs'] = similar_players['cost_diff'].abs()

    options = similar_players[similar_players['cost_diff_abs'] == similar_players['cost_diff_abs'].min()]
    above = options[options['cost_diff'] >= 0]
    below = options[options['cost_diff'] < 0]

    chosen = above.sample(1).iloc[0] if not above.empty else below.sample(1).iloc[0]
    return chosen

## 8. Run for the new players only

In [ ]:
new_rows = needed_new.copy()
for col in ['lookalike', 'lookalike_season', 'lookalike_team', 'lookalike_price']:
    new_rows[col] = None

for index, row in new_rows.iterrows():
    pick = find_lookalike(row['full_name'], row['team'], row['pos'], row['season'], row['initial_cost'], row['date'])
    if pick is not None:
        new_rows.loc[index, ['lookalike', 'lookalike_season', 'lookalike_team', 'lookalike_price']] = [
            pick['full_name'], pick['season'], pick['team'], pick['initial_cost']]

new_rows

In [ ]:
# sanity checks - big price gaps or no lookalike found
new_rows[(new_rows.initial_cost - new_rows.lookalike_price.astype(float)).abs() > 3.0]

In [ ]:
new_rows[new_rows.lookalike.isna()]

Fix any missing ones by hand (same as the Dubravka fix before), e.g.

```python
new_rows.loc[<index>, ['lookalike', 'lookalike_season', 'lookalike_team', 'lookalike_price']] = ['Name', '24-25', 'TEAM', 4.5]
```

## 9. Combine with the old lookalikes and save

In [ ]:
player_lookalikes = pd.concat([old_lookalikes, new_rows], ignore_index=True)
player_lookalikes.to_csv(NEW_LOOKALIKES, index=False)
print(f'saved {len(player_lookalikes)} rows ({len(new_rows)} new) -> {NEW_LOOKALIKES}')
player_lookalikes.season.value_counts().sort_index()